In [ ]:
# =============================================
# CUSTOM OCR MODEL FROM SCRATCH (CNN + BiLSTM + CTC)
# =============================================

import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print("=== Step 1: Setting Parameters ===")
# -------------------------------
# 1. PARAMETERS
# -------------------------------
batch_size = 2
num_epochs = 30
learning_rate = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Parameters set. Using device: {device}")
print("Step 1 completed.\n")

print("=== Step 2: Loading Labels from Excel ===")
# -------------------------------
# 2. LOAD LABELS
# -------------------------------
df = pd.read_excel(r"D:\5_Sem\Subjects\Big Data Analaytics\Project\ocr_output1.xlsx")  # columns: file_name, text
print(f"Loaded {len(df)} rows from Excel.")

df['file_name'] = df['file_name'].astype(str).str.strip()
image_folder = r"D:\5_Sem\Subjects\Big Data Analaytics\Project\trocr_dataset\images"
df['image_path'] = df['file_name'].apply(lambda x: os.path.join(image_folder, x))
df['text'] = df['text'].fillna("").astype(str)

# Remove rows where image file does not exist
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f"Filtered dataset: {len(df)} valid image-text pairs remain.")

# Create character set
all_text = "".join(df['text'].tolist())
chars = sorted(list(set(all_text)))
chars.append("<blank>")  # CTC blank token

char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
num_classes = len(chars)

print(f"Character set created with {num_classes} unique symbols.")
print("Step 2 completed.\n")

print("=== Step 3: Building Custom Dataset ===")
# -------------------------------
# 3. CUSTOM DATASET
# -------------------------------
class PrescriptionDataset(Dataset):
    def __init__(self, dataframe, char2idx):
        self.df = dataframe
        self.char2idx = char2idx

    def __len__(self):
        return len(self.df)

    def preprocess_image(self, path):
        img = Image.open(path).convert("L")
        img = np.array(img).astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)
        return img

    def encode_text(self, text):
        return [self.char2idx[c] for c in text]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.preprocess_image(row['image_path'])
        img_tensor = torch.tensor(img, dtype=torch.float)
        label = torch.tensor(self.encode_text(row['text']), dtype=torch.long)
        return img_tensor, label

print("Custom dataset class defined.")
print("Step 3 completed.\n")

print("=== Step 4: Defining Collate Function ===")
# -------------------------------
# 4. COLLATE FUNCTION FOR CTC (DYNAMIC PADDING)
# -------------------------------
def collate_fn(batch):
    images = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    
    max_h = max([img.shape[1] for img in images])
    max_w = max([img.shape[2] for img in images])
    
    padded_images = []
    for img in images:
        c, h, w = img.shape
        pad_h = max_h - h
        pad_w = max_w - w
        padded_img = np.pad(img, ((0,0),(0,pad_h),(0,pad_w)), mode='constant', constant_values=0)
        padded_images.append(padded_img)
    
    images_np = np.array(padded_images)
    images_tensor = torch.from_numpy(images_np).float()
    
    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels = torch.cat(labels)
    
    return images_tensor, labels, label_lengths

dataset = PrescriptionDataset(df, char2idx)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
print(f"DataLoader initialized with {len(dataset)} samples and batch size {batch_size}.")
print("Step 4 completed.\n")

print("=== Step 5: Defining OCR Model (CNN + BiLSTM + CTC) ===")
# -------------------------------
# 5. MODEL DEFINITION
# -------------------------------
class OCRModel(nn.Module):
    def __init__(self, num_classes):
        super(OCRModel, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64,128,3,1,1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        
        self.rnn = nn.LSTM(
            input_size=128,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )
        
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = nn.functional.adaptive_avg_pool2d(x, (1, x.size(3)))
        x = x.squeeze(2)
        x = x.permute(0,2,1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        return x

model = OCRModel(num_classes).to(device)
ctc_loss = nn.CTCLoss(blank=char2idx["<blank>"])
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Model initialized with {num_classes} output classes.")
print("Step 5 completed.\n")

print("=== Step 6: Starting Training Loop ===")
# -------------------------------
# 6. TRAINING LOOP
# -------------------------------
for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} started ---")
    model.train()
    total_loss = 0
    for imgs, labels, label_lengths in dataloader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(imgs)
        log_probs = logits.log_softmax(2).permute(1,0,2)
        
        input_lengths = torch.full(size=(logits.size(0),), fill_value=logits.size(1), dtype=torch.long)
        
        loss = ctc_loss(log_probs, labels, input_lengths, label_lengths)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    print(f"Epoch {epoch+1} completed. Average Loss: {total_loss/len(dataloader):.4f}")

print("Training completed successfully.")
print("Step 6 completed.\n")

print("=== Step 7: Defining Inference Function ===")
# -------------------------------
# 7. INFERENCE FUNCTION
# -------------------------------
def decode(preds):
    preds = preds.argmax(2)
    texts = []
    for p in preds:
        text = ""
        prev = -1
        for idx in p:
            if idx != prev and idx != char2idx["<blank>"]:
                text += idx2char[idx.item()]
            prev = idx
        texts.append(text)
    return texts

def predict(image_path):
    model.eval()
    print(f"Running inference on: {image_path}")
    img = dataset.preprocess_image(image_path)
    img_tensor = torch.tensor(img, dtype=torch.float).unsqueeze(0).to(device)
    
    _, _, h, w = img_tensor.shape
    if w % 4 != 0:
        pad_w = 4 - (w % 4)
        img_tensor = nn.functional.pad(img_tensor, (0,pad_w))
    
    with torch.no_grad():
        logits = model(img_tensor)
    text = decode(logits)[0]
    return text

print("Inference functions defined.")
print("Step 7 completed.\n")

print("=== Step 8: Testing on New Image ===")
# -------------------------------
# 8. TEST ON NEW IMAGE
# -------------------------------
test_path = r"D:\5_Sem\Subjects\Big Data Analaytics\Project\trocr_dataset\images\new_prescription.png"
predicted_text = predict(test_path)
print("Predicted Text:", predicted_text)
print("Step 8 completed. OCR pipeline executed successfully.")


=== Step 1: Setting Parameters ===
Parameters set. Using device: cpu
Step 1 completed.

=== Step 2: Loading Labels from Excel ===
Loaded 129 rows from Excel.
Filtered dataset: 129 valid image-text pairs remain.
Character set created with 122 unique symbols.
Step 2 completed.

=== Step 3: Building Custom Dataset ===
Custom dataset class defined.
Step 3 completed.

=== Step 4: Defining Collate Function ===
DataLoader initialized with 129 samples and batch size 2.
Step 4 completed.

=== Step 5: Defining OCR Model (CNN + BiLSTM + CTC) ===
Model initialized with 122 output classes.
Step 5 completed.

=== Step 6: Starting Training Loop ===

--- Epoch 1/30 started ---
